# AI-Powered Customer Churn Prediction & Exploratory Data Analysis
**Author:** Data Analyst & Machine Learning Portfolio Project  
**Objective:** Identify key drivers of customer attrition, clean raw customer records, build leakage-free predictive models, and extract actionable retention strategies.

## 1. Environment Setup & Data Acquisition
We load necessary libraries for data manipulation, statistical analysis, and machine learning pipelines.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set display aesthetics
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Import project cleaning pipeline
sys.path.append(os.path.abspath('..'))
from src.data_cleaning import clean_customer_data

df, quality_report = clean_customer_data()
print("Data cleaning completed successfully.")
display(pd.DataFrame([quality_report]))

### Data Cleaning Analysis & Justification:
The raw dataset contained 7,043 customer accounts. Key data quality challenges addressed:
1. **Whitespace Total Charges:** 11 customers with 0 tenure had blank space strings `' '` in `TotalCharges`. Because these represent new accounts that have not completed their first billing cycle, their `TotalCharges` were set to `$0.00` rather than dropping the rows.
2. **Type Casting:** `TotalCharges` was safely converted from object to float.
3. **Zero Loss of Critical Observations:** We preserved all customer records, maintaining dataset integrity.

## 2. Target Variable Analysis: Customer Churn Distribution

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pcts = df['Churn'].value_counts(normalize=True) * 100

plt.figure(figsize=(7, 4))
sns.barplot(x=churn_counts.index, y=churn_counts.values, palette=['#3B82F6', '#EF4444'])
plt.title(f"Overall Churn Distribution (Churn Rate: {churn_pcts['Yes']:.1f}%)", fontsize=14, fontweight='bold')
plt.ylabel("Customer Count")
for i, count in enumerate(churn_counts.values):
    plt.text(i, count + 100, f"{count:,} ({churn_pcts.iloc[i]:.1f}%)", ha='center', fontweight='bold')
plt.ylim(0, max(churn_counts.values) * 1.15)
plt.show()

### Target Distribution Insights:
Approximately **26.5%** of the customer base has churned. This demonstrates moderate class imbalance (approx. 3:1 ratio of retained to churned customers). As a result:
- Standard accuracy alone is an insufficient metric.
- We must prioritize **Recall, Precision, F1-Score, and ROC-AUC** during modeling to ensure churners are captured.

## 3. Exploratory Data Analysis: Key Categorical Churn Drivers
We investigate churn proportions across contracts, internet architectures, and payment methods.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Churn by Contract
contract_churn = df.groupby('Contract')['Churn'].apply(lambda s: (s == 'Yes').mean() * 100)
sns.barplot(x=contract_churn.index, y=contract_churn.values, ax=axes[0], palette='Blues_r')
axes[0].set_title('Churn Rate by Contract Type (%)', fontweight='bold')
axes[0].set_ylabel('Churn Rate (%)')
for i, v in enumerate(contract_churn.values):
    axes[0].text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')

# Churn by Internet Service
inet_churn = df.groupby('InternetService')['Churn'].apply(lambda s: (s == 'Yes').mean() * 100)
sns.barplot(x=inet_churn.index, y=inet_churn.values, ax=axes[1], palette='Oranges_r')
axes[1].set_title('Churn Rate by Internet Service (%)', fontweight='bold')
for i, v in enumerate(inet_churn.values):
    axes[1].text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')

# Churn by Payment Method
pay_churn = df.groupby('PaymentMethod')['Churn'].apply(lambda s: (s == 'Yes').mean() * 100).sort_values(ascending=False)
sns.barplot(x=pay_churn.values, y=pay_churn.index, ax=axes[2], palette='Reds_r')
axes[2].set_title('Churn Rate by Payment Method (%)', fontweight='bold')
axes[2].set_xlabel('Churn Rate (%)')

plt.tight_layout()
plt.show()

### Categorical Driver Takeaways:
1. **Contract Type:** Month-to-month contracts experience over 40% churn, while 2-year contracts show under 3%. This is the single strongest associative categorical feature.
2. **Internet Service:** Fiber optic users churn at roughly double the rate of DSL customers. Since fiber is a premium offering with higher monthly charges, this suggests potential quality-of-service issues or price sensitivity.
3. **Payment Method:** Electronic check customers churn at over 40%, whereas automated bank transfers and credit cards churn at ~15-18%.

## 4. Continuous Feature Analysis: Tenure & Monthly Charges

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Tenure distribution by Churn
sns.kdeplot(data=df, x='tenure', hue='Churn', common_norm=False, fill=True, palette=['#3B82F6', '#EF4444'], ax=axes[0])
axes[0].set_title('Tenure Density: Churned vs Retained', fontweight='bold')
axes[0].set_xlabel('Tenure (Months)')

# Monthly Charges distribution by Churn
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', palette=['#3B82F6', '#EF4444'], ax=axes[1])
axes[1].set_title('Monthly Charges Distribution by Churn Status', fontweight='bold')
axes[1].set_ylabel('Monthly Charges ($)')

plt.tight_layout()
plt.show()

### Continuous Feature Insights:
- **Tenure:** Churn hazard is concentrated heavily in the first 0-12 months of customer lifespan. Once a customer reaches 24+ months, retention increases dramatically.
- **Monthly Charges:** The median monthly charge of churned customers is markedly higher (~$80/month) than retained customers (~$65/month).

## 5. Machine Learning Pipeline: Leakage-Free Classification
We train and benchmark Logistic Regression and Random Forest models using `ColumnTransformer` and `Pipeline`.

In [ ]:
from src.model import train_and_evaluate_models

model_metrics = train_and_evaluate_models(df)
print(f"Model training completed. Best model: {model_metrics['best_model_name']}")

# Format comparison table
comp_rows = []
for mname, mdata in model_metrics['models'].items():
    comp_rows.append({
        'Model': mname,
        'Accuracy': f"{mdata['accuracy']*100:.2f}%",
        'Precision': f"{mdata['precision']*100:.2f}%",
        'Recall': f"{mdata['recall']*100:.2f}%",
        'F1-Score': f"{mdata['f1']*100:.2f}%",
        'ROC-AUC': f"{mdata['roc_auc']:.4f}"
    })
display(pd.DataFrame(comp_rows))

## 6. Strategic Business Conclusion & Synthesis

### Core Findings:
1. **Contract Structure:** Month-to-month contracts are the predominant risk factor. Converting accounts to 1- or 2-year commitments is the most effective operational intervention.
2. **Onboarding Hazard:** Customers are most vulnerable to churn during their first 12 months. Focused onboarding journeys and milestone satisfaction surveys can mitigate early departures.
3. **Payment Automation:** Manual electronic checks create billing friction and double the churn rate compared to automated card or bank billing.
4. **Bundling Support:** Adding Tech Support and Online Security reduces churn substantially, providing high ROI for retention bundling programs.

*Disclaimer:* All identified drivers represent empirical statistical associations and should be validated through randomized controlled A/B testing before enterprise-wide policy changes.